# 01 — Data Exploration

Load the Canada 2026 Race session, verify available telemetry channels, and confirm
active aero channel encoding, brake channel type, and lateral acceleration availability.

Key question for 2026: does FastF1 still expose active aero state via the `DRS` column
with the same >10 threshold, or is there a new dedicated channel?

In [1]:
import sys
import os
sys.path.insert(0, '..')

import fastf1
import fastf1.plotting
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

os.makedirs('../cache', exist_ok=True)
fastf1.Cache.enable_cache('../cache')
fastf1.plotting.setup_mpl()

## Load session

In [ ]:
session = fastf1.get_session(2026, 'Canada', 'R')
session.load(telemetry=True, weather=True)
print('Drivers:', list(session.drivers))

## Pick a driver and inspect a lap's telemetry channels

In [3]:
# Use the race winner for initial exploration
winner_abbr = session.results.iloc[0]['Abbreviation']
print('Winner:', winner_abbr)

laps = session.laps.pick_drivers(winner_abbr)
# Pick a representative mid-race lap
lap = laps.iloc[len(laps)//2]
tel = lap.get_telemetry()

print('\nTelemetry columns:')
print(tel.columns.tolist())
print('\nShape:', tel.shape)
tel.head()

Winner: LEC

Telemetry columns:
['Date', 'SessionTime', 'DriverAhead', 'DistanceToDriverAhead', 'Time', 'RPM', 'Speed', 'nGear', 'Throttle', 'Brake', 'DRS', 'Source', 'Distance', 'RelativeDistance', 'Status', 'X', 'Y', 'Z']

Shape: (632, 18)


,Date,SessionTime,DriverAhead,DistanceToDriverAhead,Time,RPM,Speed,nGear,Throttle,Brake,DRS,Source,Distance,RelativeDistance,Status,X,Y,Z
2,2024-09-01 13:40:39.512,0 days 01:32:55.593000,,147.713056,0 days 00:00:00,11330.499981,317.449999,8,99.0,False,0,interpolation,0.020732,0.000004,OnTrack,-1378.266965,-729.972166,1871.149300
3,2024-09-01 13:40:39.649,0 days 01:32:55.730000,,147.713056,0 days 00:00:00.137000,11371.599962,318.819999,8,99.0,False,0,pos,12.162709,0.002119,OnTrack,-1359.000000,-508.000000,1872.000000
4,2024-09-01 13:40:39.667,0 days 01:32:55.748000,,147.713056,0 days 00:00:00.155000,11377.000000,319.000000,8,99.0,False,0,car,13.759722,0.002397,OnTrack,-1356.847896,-483.221094,1872.106897
5,2024-09-01 13:40:39.868,0 days 01:32:55.949000,4,147.713056,0 days 00:00:00.356000,11429.000000,320.000000,8,99.0,False,0,car,31.626389,0.005510,OnTrack,-1338.762702,-275.273136,1873.222618
6,2024-09-01 13:40:40.049,0 days 01:32:56.130000,4,148.014722,0 days 00:00:00.537000,11446.345827,321.508333,8,99.0,False,0,pos,47.798605,0.008327,OnTrack,-1328.000000,-152.000000,1874.000000


## Verify active aero channel encoding

In 2026 the active aero system replaces DRS. FastF1 is expected to map active-aero
state to the same `DRS` column — but we verify this explicitly. Values > 10 should
indicate straight mode (active aero open); ≤ 10 = corner mode.

Check for any new 2026-specific channels (wing angle, active aero position) that
might give a cleaner signal.

In [ ]:
# Check for active aero / DRS channel
if 'DRS' in tel.columns:
    print('DRS unique values:', sorted(tel['DRS'].unique()))
    print('DRS dtype:', tel['DRS'].dtype)
    n_open = (tel['DRS'] > 10).sum()
    print(f'\n"DRS" > 10 rows: {n_open} / {len(tel)} ({100*n_open/len(tel):.1f}%)')
    print('FastF1 2024 convention: 0=closed, 12=enabled, 14=open')
    print('2026 expectation: same channel, same values — VERIFY this matches')
else:
    print('WARNING: No DRS column found.')

# Check for any new 2026 active-aero-specific channels
possible_new = ['ActiveAero', 'active_aero', 'WingAngle', 'wing_angle',
                'FrontWing', 'RearWing', 'AeroMode', 'aero_mode']
new_found = [c for c in possible_new if c in tel.columns]
print(f'\nNew active-aero channels: {new_found if new_found else "none found"}')
print('All columns:', tel.columns.tolist())

# Plot active aero state vs distance to confirm zone boundaries
fig, ax = plt.subplots(figsize=(14, 3))
if 'DRS' in tel.columns:
    ax.plot(tel['Distance'], tel['DRS'], label='DRS/active aero channel')
    ax.axhline(10, color='r', linestyle='--', label='threshold (>10 = straight mode)')
ax.set_xlabel('Distance (m)')
ax.set_ylabel('Active aero channel value')
ax.set_title(f'Active aero state vs distance — {winner_abbr}, Canada 2026')
ax.legend()
plt.tight_layout()
plt.savefig('../results/figures/01_active_aero_encoding.png', dpi=150)
plt.show()

## Verify brake channel

In [ ]:
print('Brake dtype:', tel['Brake'].dtype)
print('Brake unique (first 10):', tel['Brake'].unique()[:10])
print('Throttle dtype:', tel['Throttle'].dtype)
print('Speed dtype:', tel['Speed'].dtype)

## Check for lateral acceleration channel

In [ ]:
lat_candidates = ['lateral_acceleration', 'LateralAcceleration', 'lateral_g', 'LatG', 'lat_g', 'ay', 'Ay']
found = [c for c in lat_candidates if c in tel.columns]
print('Direct lateral g channels found:', found)

pos_channels = [c for c in tel.columns if c in ['X', 'Y', 'Z']]
print('Position channels found:', pos_channels)

if not found:
    print('\nNo direct lateral g — will compute from X/Y GPS position in Phase 4.')
    if 'X' in tel.columns and 'Y' in tel.columns:
        print('GPS position available — computed lat g is feasible.')
    else:
        print('WARNING: No GPS position either — ClA estimation will need an alternative.')

## Weather data check

In [ ]:
weather = session.weather_data
print('Weather columns:', weather.columns.tolist())
print(weather[['AirTemp', 'Pressure', 'Humidity']].describe())

## Full telemetry overview for a single lap

In [ ]:
fig, axes = plt.subplots(5, 1, figsize=(14, 12), sharex=True)

axes[0].plot(tel['Distance'], tel['Speed'])
axes[0].set_ylabel('Speed (km/h)')

axes[1].plot(tel['Distance'], tel['Throttle'], color='g')
axes[1].set_ylabel('Throttle (%)')

axes[2].plot(tel['Distance'], tel['Brake'].astype(float), color='r')
axes[2].set_ylabel('Brake')

axes[3].plot(tel['Distance'], (tel['DRS'] > 10).astype(int), color='purple')
axes[3].set_ylabel('Straight mode')
axes[3].set_ylim(-0.1, 1.1)

if 'X' in tel.columns:
    axes[4].set_visible(False)
    axes[4] = fig.add_subplot(5, 1, 5)
    axes[4].plot(tel['X'], tel['Y'], lw=0.5)
    axes[4].set_aspect('equal')
    axes[4].set_xlabel('X (m)')
    axes[4].set_ylabel('Y (m)')
else:
    axes[4].set_ylabel('(no GPS)')
    axes[4].set_xlabel('Distance (m)')

fig.suptitle(f'Telemetry overview — {winner_abbr} — Canada 2026')
plt.tight_layout()
plt.savefig('../results/figures/01_telemetry_overview.png', dpi=150)
plt.show()

## Summary

Update this cell after running with Canada 2026 data:

- Active aero channel: **TBD** — verify `DRS` column still present and values 0/12/14
- Brake channel: **TBD** — expected bool (same as 2024)
- Lateral g: **TBD** — expected no direct channel, compute from GPS X/Y
- New 2026 channels: **TBD** — check for WingAngle, ActiveAero, etc.
- Weather: AirTemp and Pressure available — verify Montreal conditions

If `DRS` encoding differs from 0/12/14, update `DRS_OPEN_THRESHOLD` in
`src/segments.py` before running notebooks 02–05.